# Context Engineering for Agents

## Scenario: Acme EU payment incident

The incident assistant needs enough context to decide its next investigation step—without a full transcript, another tenant’s data, or a poisoned runbook. This notebook builds that context packet deterministically, then explains how to connect the same design to a model or state graph.

**Safety:** no external model, data store, or action is invoked. `lab.py` simulates context selection and preserves application-side trust and tenant checks.

## The context-routing loop

![Context routing diagram](assets/context-routing.svg)

The system controls which tokens reach a model for each decision. System policy, identity, state, conversation, tool evidence, and external memory are different inputs with different trust and scope. Context selection happens before model inference; cross-tenant or poisoned data is quarantined before assembly.

## 1. Context is a decision-time resource

Prompt engineering improves instructions. **Context engineering** decides the whole information configuration for one inference: stable instructions, dynamic task data, tool results, environment state, thread history, and retrieved memory. The target is the smallest sufficient high-signal packet—not the longest possible prompt.

A long context window is capacity, not a permission boundary or a memory system. External memory lives outside the window and must be retrieved by scope, freshness, relevance, and purpose.

In [ ]:
from lab import Request, context_view, explain_packet, route_context

request = Request('acme', 'support-17', 'Why are EU checkout payments failing?', 'investigate')
packet = route_context(request)
print('Selected:', context_view(packet))
print('Tokens:', packet.token_estimate)
print('Quarantined:', packet.quarantined)
print('Dropped:', packet.dropped)
print('\nCompressed handoff:\n', packet.summary)

## 2. Context layers and trust

- **System instructions** are stable, application-owned policy: evidence standards, permissions, output requirements, and stop behavior.
- **Dynamic context** is selected for the current phase: symptoms and identity during triage; verified tool/document evidence during investigation; approved risk policy before recommendation.
- **Environment state** contains phase, budget, approval, and evidence handles. It is not editable via chat text.
- **Tool context** must be typed, small, fresh, and attributable.
- **Conversation state** is useful only while relevant to the active task.
- **External memory** persists across threads, so it needs tenant/user namespaces, validation, revocation, and a retrieval policy.

In the lab, a premium-SLA memory is scoped to Acme; Globex data is excluded even though it has high relevance; an injection-bearing runbook is quarantined despite being relevant.

In [ ]:
# Triage loads a smaller starter packet; investigation loads evidence just in time.
triage = route_context(Request('acme', 'support-17', 'Payments failing?', 'triage'))
investigate = route_context(Request('acme', 'support-17', 'Payments failing?', 'investigate'))
print('triage items:', [item_id for item_id, _, _ in context_view(triage)])
print('investigate items:', [item_id for item_id, _, _ in context_view(investigate)])
assert 'health-1' not in [item_id for item_id, _, _ in context_view(triage)]
assert 'health-1' in [item_id for item_id, _, _ in context_view(investigate)]
assert 'other-tenant' not in [item_id for item_id, _, _ in context_view(investigate)]
assert 'poisoned-runbook' in investigate.quarantined

## 3. Just-in-time routing, pruning, and compression

The agent does not need every runbook and log at triage. It starts with a stable packet, calls a bounded tool, validates the result, then routes the next evidence item. This progressive disclosure reduces token use and stale-information distraction, but it needs a robust fallback when retrieval fails.

**Pruning** removes low-value messages. **Summarization** rewrites history. **Structured compression** preserves fields required for recovery: decision, evidence IDs, state, constraints, and open question. **Caching** avoids rebuilding stable packets, but its key must include tenant, identity, phase/task, policy, and source versions. Never share a cache by textual similarity alone.

In [ ]:
first = route_context(request)
same = route_context(request)
other_tenant = route_context(Request('globex', 'support-17', 'Why are EU checkout payments failing?', 'investigate'))
print('repeat cache key:', first.cache_key == same.cache_key)
print('tenant-separated cache key:', first.cache_key != other_tenant.cache_key)
print(explain_packet(first)['summary'])

## 4. Context poisoning and isolation

A web page, retrieved document, tool result, or old summary can contain instructions intended to alter behavior. It is data—not authority. Filter scope before prompt assembly, annotate provenance and trust, delimit untrusted content, keep authorization in code, and give the agent no broad instruction-editing or permission-granting tool. A detector can help triage, but it cannot replace isolation.

The failure case here is instructive: a poisoned Acme runbook is more relevant than some safe artifacts. The correct behavior is quarantine, not inclusion. Relevance is not trust. Likewise, a trusted Globex document never enters an Acme packet: relevance is not authorization.

## Production checklist and exercises

- Version and test context assembly separately from prompts.
- Scope every item by tenant/user/project and check that scope before assembly and caching.
- Attach source ID, timestamp, sensitivity, trust, and version metadata to tool/retrieval results.
- Use token budgets and structured compaction that preserves decisions, evidence, constraints, and unresolved gaps.
- Evaluate answer quality *and* selection quality: recall of needed evidence, irrelevant-token rate, cross-tenant block rate, poisoned-content quarantine, cache isolation, latency, and token cost.

**Exercises:** add a stale trusted document and freshness filter; add an approval state that cannot be forged through user text; build a summary schema for a 100-turn incident; compare full transcript vs structured summary + JIT retrieval; sketch a LangGraph state containing `context_packet`, `evidence_handles`, and `policy_version`.

## Sources

- [Anthropic: Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
- [LangGraph memory overview](https://docs.langchain.com/oss/python/concepts/memory)
- [OpenAI context-personalization cookbook](https://developers.openai.com/cookbook/examples/agents_sdk/context_personalization)
- [LLM autonomous agents survey](https://arxiv.org/abs/2308.11432)
- [OWASP prompt injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/)